# Calculating Correlations of Teleconnection Indices with Detrended Standardized Climate Anomalies in 13 Regions
## By Landon Moeller

### Importing Packages

In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from ipywidgets import interact, interactive, fixed, interact_manual
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import math

### Opening Datasets

In [3]:
nh_ds = xr.open_dataset("../Data/NH_tele_cli_anoms_detrended_1950-2024.nc")
sh_ds = xr.open_dataset("../Data/SH_tele_cli_anoms_detrended_1950-2024.nc")
global_ds = xr.open_dataset("../Data/Global_tele_cli_anoms_detrended_1950-2024.nc")
nh_ds

<xarray.Dataset> Size: 497kB
Dimensions:                                  (year: 75, month: 12)
Coordinates:
  * year                                     (year) int64 600B 1950 ... 2024
  * month                                    (month) int64 96B 1 2 3 ... 11 12
    time                                     (year, month) datetime64[ns] 7kB ...
Data variables: (12/68)
    TNA                                      (year, month) float64 7kB ...
    WPO                                      (year, month) float64 7kB ...
    WP                                       (year, month) float64 7kB ...
    SCA                                      (year, month) float64 7kB ...
    EAWR                                     (year, month) float64 7kB ...
    EA                                       (year, month) float64 7kB ...
    ...                                       ...
    TP_Anomaly_Western_Australia_Detrended   (year, month) float64 7kB ...
    TP_StdAnom_Western_Australia_Detrended   (year, month) float64 7kB ...
    T2M_Anomaly_Northeast_Detrended          (year, month) float64 7kB ...
    T2M_StdAnom_Northeast_Detrended          (year, month) float64 7kB ...
    TP_Anomaly_Northeast_Detrended           (year, month) float64 7kB ...
    TP_StdAnom_Northeast_Detrended           (year, month) float64 7kB ...
Attributes:
    Name:                   NH Teleconnections and Regional Climate Anomalies...
    Teleconnection_source:  Climate Prediction Center
    Anomaly_source:         ECMWF Reanalysis v5 (ERA5)

### Defining Calculation and Plotting Functions

In [12]:
nh_indices = ['TNA', 'WPO', 'WP', 'SCA', 'EAWR', 'EA', 'ADI', 'AMO',
              'AO', 'EPNP', 'EPO', 'NAO', 'NOI', 'PDO', 'PNA', 'POL']

sh_indices  = ['SAOD', 'TSA', 'SPOD', 'TPI', 'SOI', 'AAO']

global_indices = ['IPO', 'IOD', 'EMI', 'QBO', 'MEI', 'AAM', 'ENSO_34']

month_names = {
        1: "January", 2: "February", 3: "March", 4: "April",
        5: "May", 6: "June", 7: "July", 8: "August",
        9: "September", 10: "October", 11: "November", 12: "December"
    }

# Determining which teleconnection list belongs to the dataset
def get_indices(ds):
    if 'TNA' in ds.variables: return nh_indices
    if 'SAOD' in ds.variables: return sh_indices
    if 'IPO' in ds.variables: return global_indices
    return nh_indices

def calc_tele_stdanom_corrs(ds, region, variable, month=None):

    indices = get_indices(ds)
    var_name = f"{variable}_StdAnom_{region}_Detrended"

    if var_name not in ds:
        print(f"Variable not found: {var_name}")
        return

    # Selecting either a single month or all months stacked together
    if month is not None:
        ds_sel = ds.sel(month=month)
        period = f"Month = {month}"
    else:
        ds_sel = ds.stack(sample=('year', 'month'))
        period = "All Months"

    y = ds_sel[var_name].values.ravel()

    print(f"\nCorrelations for {variable} in {region}")
    print("** = p < 0.01 (very statistically significant)")
    print("* = p < 0.05 (statistically significant)\n")

    results = []

    for idx in indices:
        if idx not in ds_sel:
            continue

        x = ds_sel[idx].values.ravel()

        # Removing NaNs
        valid = ~np.isnan(x) & ~np.isnan(y)
        n_valid = valid.sum()

        if n_valid < 10:
            print(f"WARNING: {idx} has only {n_valid} valid data pairs in {period}!\n")
            continue

        x_valid = x[valid]
        y_valid = y[valid]

        # Computing correlation and regression statistics
        r, p = pearsonr(x_valid, y_valid)

        model = LinearRegression().fit(x_valid.reshape(-1, 1), y_valid)
        r2 = r2_score(y_valid, model.predict(x_valid.reshape(-1, 1)))

        results.append((idx, r, p, r2))

    # Sorting teleconnections by strongest absolute correlation
    results.sort(key=lambda t: abs(t[1]), reverse=True)

    print(f"{'Index':<6} {'r':>7} {'R²':>11} {'p-value':>15} {'Corr. Strength':>19}")
    print("-" * 62)

    for idx, r, p, r2 in results:
        sig = "**" if p < 0.01 else "*" if p < 0.05 else " "
        strength = "(STRONG)" if abs(r) >= 0.5 else "(MODERATE)" if abs(r) >= 0.3 else " "
        print(f"{idx:<6} {r:10.4f} {r2:10.4f} {p:14.4e} {sig:>2}    {strength}")

def plot_teleconnection_scatter(
    ds,
    tele_x,
    var_y,
    anomaly_var,
    month=None,
    cmap='coolwarm',
    upper_label='',
    lower_label='',
):

    # Selecting either one month or all months combined
    if month is not None:
        ds_sel = ds.sel(month=month)
        month_str = f"{month_names[month]}"
    else:
        ds_sel = ds.stack(sample=("year", "month"))
        month_str = "All Months"

    tele  = ds_sel[tele_x].values.ravel()
    var   = ds_sel[var_y].values.ravel()
    color = ds_sel[anomaly_var].values.ravel()

    # Extracting year labels for annotation purposes
    if 'year' in ds_sel.coords:
        years = ds_sel['year'].values.ravel()
    elif 'sample' in ds_sel.dims and 'year' in ds.coords:
        years = ds_sel.coords['year'].values
    else:
        years = np.arange(len(tele))

    # Removing NaNs
    valid = ~np.isnan(tele) & ~np.isnan(var) & ~np.isnan(color)
    tele = tele[valid]
    var = var[valid]
    color = color[valid]
    years = years[valid]

    x_name = ds_sel[tele_x].attrs.get("Name", tele_x)
    y_name = ds_sel[var_y].attrs.get("Name", var_y)
    var_name = ds_sel[anomaly_var].attrs.get("Name", anomaly_var)
    region = ds_sel[anomaly_var].attrs.get("Region", anomaly_var)
    units = ds_sel[anomaly_var].attrs.get("units", anomaly_var)

    x_max_abs = np.max(np.abs(tele)) if len(tele) > 0 else 1
    y_max_abs = np.max(np.abs(var))  if len(var)  > 0 else 1

    xlim_abs = math.ceil(x_max_abs)
    ylim_abs = math.ceil(y_max_abs)

    xlim = (-xlim_abs, xlim_abs)
    ylim = (-ylim_abs, ylim_abs)

    # Fitting linear regression
    x_2d = tele.reshape(-1, 1)
    y_2d = var.reshape(-1, 1)
    model = LinearRegression()
    model.fit(x_2d, y_2d)
    var_pred = model.predict(x_2d)

    pearson_r, pearson_p = pearsonr(tele, var)
    r2 = r2_score(var, var_pred)

    print(f'Pearson Correlation (r): {pearson_r:.4f}')
    print(f'Pearson p-value: {pearson_p:.4e}')
    print(f'R² value: {r2:.4f}')

    fig, ax = plt.subplots(figsize=(9, 6), dpi=400)

    # Highlighting the most extreme positive and negative anomalies
    if len(color) >= 6:
        sorted_idx = np.argsort(color)
        neg_extremes = sorted_idx[:10]
        pos_extremes = sorted_idx[-10:][::-1]
        extremes = np.concatenate([neg_extremes, pos_extremes])

        ax.scatter(tele[extremes], var[extremes], c='black', marker='x', s=75, linewidths=3, zorder=3)

    # Plotting scatter points colored by anomaly magnitude
    sc = ax.scatter(tele, var, c=color, cmap=cmap, marker='x', s=60, alpha=1, vmin=-ylim_abs, vmax=ylim_abs, zorder=5)

    # Plotting regression line
    ax.plot(tele, var_pred, lw=3, color='black', zorder=10)

    ax.axhline(0, color='gray', lw=1.1, alpha=0.7, zorder=0)
    ax.axvline(0, color='gray', lw=1.1, alpha=0.7, zorder=0)

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.grid(True, alpha=0.3, linestyle='--', zorder=0)

    ax.set_xlabel(f"{x_name} ({tele_x})", fontsize=11, fontweight='bold')
    ax.set_ylabel(f"{y_name}", fontsize=11, fontweight='bold')

    ax.set_title(f"{tele_x} vs {var_name} ({region})", fontsize=13, fontweight='bold')

    cbar = plt.colorbar(sc, ax=ax, pad=0.015, aspect=30)
    cbar.set_label(f"{var_name} ({units})", fontsize=10)

    # Adding correlation statistics and the month label to the figure
    plt.figtext(0.1355, 0.811, f'R = {pearson_r:.3f}\nR² = {r2:.3f}',
                color='dimgray', fontsize=12, fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.5, edgecolor='lightgray'))
    plt.figtext(0.761, 0.8425, f'{month_str}',
                ha='right', color='dimgray', fontsize=12, fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.5, edgecolor='lightgray'))

    if len(color) >= 6:

        # Converting indices into readable month/year strings
        def get_date(i):
            if month is not None:
                return f"{month:02d}/{int(ds_sel['year'].values.ravel()[valid][i])}"
            else:
                m = int(ds_sel.coords['month'].values[valid][i])
                y = int(ds_sel.coords['year'].values[valid][i])
                return f"{m:02d}/{y}"

        cmap = sc.cmap
        sorted_pairs = np.argsort(color)
        upper_pairs = sorted_pairs[-10:][::-1]
        lower_pairs  = sorted_pairs[:10]

        upper_occurrences = []
        for i in upper_pairs:
            anom = color[i]
            norm = (anom + ylim_abs) / (2 * ylim_abs)
            col = cmap(norm)[:3]
            date_str = get_date(i)
            upper_occurrences.append((date_str, col))

        lower_occurrences = []
        for i in lower_pairs:
            anom = color[i]
            norm = (anom + ylim_abs) / (2 * ylim_abs)
            col = cmap(norm)[:3]
            date_str = get_date(i)
            lower_occurrences.append((date_str, col))

        # Listing the strongest positive anomaly dates
        fig.text(0.14, 0.035, f"{upper_label}", ha='right', fontsize=9, fontweight='bold', color='black')

        x = 0.145
        for date_str, col in upper_occurrences:
            fig.text(x, 0.034, f"{date_str}", ha='left', fontsize=9, color=col)
            x += len(date_str) * 0.009 + 0.01

        # Listing the strongest negative anomaly dates
        fig.text(0.14, 0.005, f"{lower_label}", ha='right', fontsize=8.8, fontweight='bold', color='black')

        x = 0.145
        for date_str, col in lower_occurrences:
            fig.text(x, 0.004, f"{date_str}", ha='left', fontsize=8.8, color=col)
            x += len(date_str) * 0.009 + 0.01

        plt.subplots_adjust(bottom=0.14)

    plt.savefig(f"{month_str}_{tele_x}_{anomaly_var}.png".replace(" ", "_"), dpi=400, bbox_inches='tight')
    plt.show()

### Calculating Correlations Widget

Running this widget prints out the Pearson r, R$^2$, and p-value for the selected region, teleconnection index, variable, and month.

In [13]:
dataset_options = {
    'Northern Hemisphere': 'nh_ds',
    'Southern Hemisphere': 'sh_ds',
    'Global': 'global_ds'
}

region_options = [
    'Midwest', 'Southern_Plains', 'Northeast', 'Western_Europe', 'Eastern_Europe', 'Black_Sea', 'India',
    'East_Asia', 'Western_Australia', 'Eastern_Australia', 'Northern_Brazil', 'Southern_Brazil', 'Argentina'
]

variable_options = {'Std Temp Anomaly - Detrended': 'T2M', 'Std Precip Anomaly - Detrended': 'TP'}

month_options = [('All Months', None)] + [(month_names.get(m, str(m)), m) for m in range(1,13)]

ds_dropdown = widgets.Dropdown(
    options=dataset_options.keys(),
    description='Dataset:',
    layout={'width': 'max-content'}
)

region_dropdown = widgets.Dropdown(
    options=region_options,
    description='Region:',
    layout={'width': 'max-content'}
)

var_dropdown = widgets.Dropdown(
    options=variable_options,
    description='Variable:',
    layout={'width': 'max-content'}
)

month_dropdown = widgets.Dropdown(
    options=month_options,
    description='Month:',
    layout={'width': 'max-content'}
)

run_button = widgets.Button(
    description="Run Correlations",
    button_style='success',
    icon='play'
)

output = widgets.Output()

# This runs the calculation + printout function for your desired region, variable, and month
def on_run_button_clicked(b):
    with output:
        output.clear_output()
        ds_name = dataset_options[ds_dropdown.value]
        ds = globals()[ds_name]

        region = region_dropdown.value
        variable = var_dropdown.value
        month = month_dropdown.value

        print(f"{ds_name} | {region} | {variable} | month = {month}")
        print("-"*36)

        calc_tele_stdanom_corrs(ds, region, variable, month)

run_button.on_click(on_run_button_clicked)

ui = widgets.VBox([
    ds_dropdown,
    region_dropdown,
    var_dropdown,
    month_dropdown,
    run_button,
    output
])

display(ui)

### Plotting Correlations Widget

Running this widget provides and automatically saves 13 plots for the selected region, variable, and teleconnection index.

In [15]:
dataset_options = {
    'Northern Hemisphere': nh_ds,
    'Southern Hemisphere': sh_ds,
    'Global': global_ds
}

index_map = {
    'Northern Hemisphere': nh_indices,
    'Southern Hemisphere': sh_indices,
    'Global': global_indices
}

month_options = [('All Months', None)] + [(f"{month_names.get(m, 'Month ' + str(m))}", m) for m in range(1, 13)]

ds_dropdown = widgets.Dropdown(
    options=list(dataset_options.keys()),
    description='Dataset:',
    layout={'width': 'max-content'}
)

index_dropdown = widgets.Dropdown(
    options=nh_indices,
    description='Index:',
    layout={'width': 'max-content'}
)

region_dropdown = widgets.Dropdown(
    options=region_options,
    description='Region:',
    layout={'width': 'max-content'}
)

var_dropdown = widgets.Dropdown(
    options=variable_options,
    description='Variable:',
    layout={'width': 'max-content'}
)

run_button = widgets.Button(
    description="Build Plots",
    button_style='success',
    icon='play',
)

output = widgets.Output()

def update_tele_options(change):
    selected_ds_label = change['new']
    new_indices = index_map.get(selected_ds_label, nh_indices)

    index_dropdown.options = new_indices
    index_dropdown.value = new_indices[0] if new_indices else None

ds_dropdown.observe(update_tele_options, names='value')

update_tele_options({'new': ds_dropdown.value})

# This runs the calculation + plotting function for your desired region, teleconnection index, and variable
def on_run_button_clicked(b):

    with output:
        clear_output(wait=True)

        ds = dataset_options[ds_dropdown.value]
        tele_x = index_dropdown.value
        region = region_dropdown.value
        variable = var_dropdown.value

        if variable == 'T2M':
            cmap = 'coolwarm'
            upper_label = 'Warmest: '
            lower_label = 'Coldest: '

        elif variable == 'TP':
            cmap = 'BrBG'
            upper_label = 'Wettest: '
            lower_label = 'Driest: '

        var_y = f"{variable}_StdAnom_{region}_Detrended"
        anomaly_var = var_y

        for month_label, month in month_options:

            print(f"\n{month_label}\n")

            try:
                plot_teleconnection_scatter(
                    ds=ds,
                    tele_x=tele_x,
                    var_y=var_y,
                    anomaly_var=anomaly_var,
                    month=month,
                    cmap=cmap,
                    upper_label=upper_label,
                    lower_label=lower_label
                )

            except ValueError as e:

                if "Found array with 0 sample(s)" in str(e):
                    print(f"No valid data available for {month_label}.")
                else:
                    raise

run_button.on_click(on_run_button_clicked)

ui = widgets.VBox([
    ds_dropdown,
    index_dropdown,
    region_dropdown,
    var_dropdown,
    run_button,
    output
])

display(ui)